=(recreacion-zonas-steer)
# Recreación de zonas de Steer

Esta notebook tiene como objetivo recrear la zonificación utilizada por Steer.

1. **Voronoi**: asignación rápida de segmentos censales a zonas por proximidad.
2. **Algoritmo iterativo**: (KDTree + combinaciones + chequeo de conectividad), para lo que Voronoi no pudo resolver.
3. **Algoritmo iterativo**: segunda pasada del mismo algoritmo con una tolerancia más alta, sobre lo que quedó pendiente de la pasada anterior.
4. **Mapa final**: con todas las capas para revisar cómo quedaron las zonas.
5. **Armado de zonas a mano**: corrección manual en QGIS de los segmentos mal asignados o sin asignar, y de las zonas con más de una sección censal.
6. **Asignación de departamento, sección censal y segmentos censales**: se le asigna a cada zona su correspondiente departamento, sección/es censal/es y segmentos censales.

## Setup

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from itertools import combinations
from geopandas import GeoSeries
from sklearn.neighbors import KDTree
from shapely.geometry import Polygon
import json


In [2]:
PATH_VORONOI = "../data/voronois_montevideo.gpkg"
PATH_ZONAS = "../data/zonas_filtradas.gpkg" # centroides de Steer
PATH_SEGMENTOS = "../data/segmentos_censales.gpkg"

CRS_METROS = 32721

voronoi = gpd.read_file(PATH_VORONOI).to_crs(epsg=CRS_METROS)
zonas = gpd.read_file(PATH_ZONAS).to_crs(epsg=CRS_METROS)
segmentos = gpd.read_file(PATH_SEGMENTOS).to_crs(epsg=CRS_METROS)

print(f"Zonas objetivo: {len(zonas)}")
print(f"Segmentos censales: {len(segmentos)}")
print(f"Polígonos Voronoi: {len(voronoi)}")

Zonas objetivo: 598
Segmentos censales: 1822
Polígonos Voronoi: 614


## 1. Voronoi

Cada segmento censal se asigna al polígono de Voronoi que contiene su centroide. Se disuelven los segmentos por polígono de Voronoi y se comparan esos centroides contra las zonas originales: si caen dentro de la tolerancia, la zona queda resuelta.

In [3]:
# Asignar cada segmento a su polígono de Voronoi según el centroide
segmentos["centroide"] = segmentos.geometry.centroid
segmentos_centroides = segmentos.set_geometry("centroide")
segmentos_voronoi = gpd.sjoin(segmentos_centroides, voronoi, how="left", predicate="within")
segmentos_voronoi = segmentos_voronoi.set_geometry("geometry")

# Disolver segmentos por polígono de Voronoi para armar las zonas candidatas
zonas_voronoi = segmentos_voronoi.dissolve(by="index_right")
zonas_voronoi.drop(columns=["centroide"], inplace=True, errors="ignore")
zonas_voronoi["centroide_zona"] = zonas_voronoi.geometry.centroid

In [4]:
TOLERANCIA_VORONOI = 30  # metros

zonas_en_voronoi = gpd.sjoin(zonas, zonas_voronoi, how="inner", predicate="within")
zonas_en_voronoi["centroide_zona_voronoi"] = zonas_en_voronoi.apply(
    lambda row: zonas_voronoi.loc[row["index_right"], "geometry"].centroid, axis=1)
zonas_en_voronoi["centroide_zona_voronoi"] = GeoSeries(zonas_en_voronoi["centroide_zona_voronoi"], crs=CRS_METROS)

zonas_en_voronoi["distancia"] = zonas_en_voronoi.geometry.distance(zonas_en_voronoi["centroide_zona_voronoi"])
zonas_encontradas_voronoi = zonas_en_voronoi[zonas_en_voronoi["distancia"] < TOLERANCIA_VORONOI]

print(f"Zonas encontradas con Voronoi: {len(zonas_encontradas_voronoi)} de {len(zonas)} "
      f"({len(zonas_encontradas_voronoi)/len(zonas)*100:.1f}%)")
print(f"Distancia promedio: {zonas_encontradas_voronoi['distancia'].mean():.1f}m")

Zonas encontradas con Voronoi: 369 de 598 (61.7%)
Distancia promedio: 4.6m


In [5]:
# Zonas que Voronoi no pudo resolver -> insumo para el algoritmo iterativo
zonas_missing = zonas[~zonas["ID"].isin(zonas_encontradas_voronoi["ID_left"])].copy()

# Segmentos que ya quedaron usados en las zonas resueltas por Voronoi
segmentos_usados = set()
for zona_idx in zonas_encontradas_voronoi["index_right"].unique():
    segmentos_en_zona = segmentos_voronoi[segmentos_voronoi["index_right"] == zona_idx].index
    segmentos_usados.update(segmentos_en_zona)

segmentos_disponibles = segmentos.loc[~segmentos.index.isin(segmentos_usados)].copy()

print(f"Zonas sin resolver: {len(zonas_missing)} de {len(zonas)}")
print(f"Segmentos disponibles para el algoritmo iterativo: {len(segmentos_disponibles)}")

Zonas sin resolver: 229 de 598
Segmentos disponibles para el algoritmo iterativo: 989


## 2. Algoritmo iterativo (segmentos conectados + árbol KDTree)

Para cada zona sin resolver: se buscan los `k` segmentos más cercanos con un KDTree, se prueban combinaciones de 1 a 10 segmentos, se descartan las que no forman un grupo conectado, y se queda con la combinación cuyo centroide cae dentro de la tolerancia y está más cerca de la zona. También intenta rellenar agujeros internos del polígono resultante con segmentos vecinos.

In [6]:
def es_grupo_conectado(segmentos_grupo):
    """Verifica si un grupo de segmentos forma una geometría conectada."""
    if len(segmentos_grupo) == 1:
        return True

    geoms = list(segmentos_grupo.geometry)
    n = len(geoms)

    # Grupos chicos: verificación directa (cada segmento toca a algún otro)
    if n <= 3:
        for i, geom_i in enumerate(geoms):
            if not any(
                i != j and (geom_i.touches(geom_j) or geom_i.intersects(geom_j))
                for j, geom_j in enumerate(geoms)
            ):
                return False
        return True

    # Grupos grandes: matriz de adyacencia + BFS
    matriz_adyacencia = [[False] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if geoms[i].touches(geoms[j]) or geoms[i].intersects(geoms[j]):
                matriz_adyacencia[i][j] = True
                matriz_adyacencia[j][i] = True

    if any(not any(fila) for fila in matriz_adyacencia):
        return False  # hay un segmento aislado

    visitados = {0}
    por_visitar = [0]
    while por_visitar:
        actual = por_visitar.pop()
        for j in range(n):
            if j not in visitados and matriz_adyacencia[actual][j]:
                visitados.add(j)
                por_visitar.append(j)

    return len(visitados) == n

In [7]:
def ejecutar_algoritmo_iterativo(segmentos_disponibles, zonas_missing, tolerancia,
                                  k_vecinos=15, max_grupo=10, verbose=True):
    """
    Para cada zona en zonas_missing busca la mejor combinación conectada de
    segmentos (entre 1 y max_grupo) cuyo centroide caiga a menos de `tolerancia`
    metros de la zona. Devuelve:
      - resultados: lista de dicts con cada polígono armado
      - segmentos_restantes: segmentos que no se llegaron a usar
      - zonas_restantes: zonas que no se pudieron resolver
    """
    resultados = []
    segmentos_restantes = segmentos_disponibles.copy()
    zonas_restantes = zonas_missing.copy()

    for idx, (_, zona_missing) in enumerate(zonas_missing.iterrows()):
        if zona_missing["ID"] not in zonas_restantes["ID"].values:
            continue  # ya resuelta en una iteración anterior del bucle

        if verbose and idx % 50 == 0:
            print(f"  procesando zona {idx+1}/{len(zonas_missing)} "
                  f"- segmentos restantes: {len(segmentos_restantes)}")

        if len(segmentos_restantes) < 1:
            break

        punto_missing = zona_missing.geometry
        zona_id = zona_missing["ID"]

        # Árbol de vecinos más cercanos entre los segmentos que quedan disponibles
        centroides = np.array(
            [[geom.centroid.x, geom.centroid.y] for geom in segmentos_restantes.geometry]
        )
        tree = KDTree(centroides, leaf_size=7)

        zona_point = [punto_missing.centroid.x, punto_missing.centroid.y]
        k = min(k_vecinos, len(segmentos_restantes))
        _, knn = tree.query([zona_point], k=k)
        indices_vecinos = segmentos_restantes.iloc[knn[0]].index.tolist()

        mejor_resultado = None
        mejor_distancia = float("inf")

        for j in range(1, max_grupo + 1):
            max_combinaciones = 500 if j <= 5 else 100
            combinaciones_evaluadas = 0

            for combi in combinations(indices_vecinos, j):
                if combinaciones_evaluadas >= max_combinaciones:
                    break
                combinaciones_evaluadas += 1

                grupo = segmentos_restantes.loc[list(combi)]

                if j > 1 and not es_grupo_conectado(grupo):
                    continue

                try:
                    union_geom = (
                        grupo.geometry.iloc[0] if j == 1 else grupo.geometry.union_all()
                    )
                    combi_final = list(combi)

                    # Rellenar agujeros internos con segmentos vecinos, si aplica
                    if hasattr(union_geom, "interiors") and len(union_geom.interiors) > 0:
                        segmentos_para_rellenar = []
                        for hole in union_geom.interiors:
                            hole_geom = Polygon(hole)
                            for seg_idx, seg_row in segmentos_restantes.iterrows():
                                if seg_idx not in combi_final:
                                    seg_centroid = seg_row.geometry.centroid
                                    if hole_geom.contains(seg_centroid) or hole_geom.intersects(seg_row.geometry):
                                        segmentos_para_rellenar.append(seg_idx)

                        if segmentos_para_rellenar:
                            combi_candidato = list(combi) + segmentos_para_rellenar
                            grupo_con_relleno = segmentos_restantes.loc[combi_candidato]
                            if es_grupo_conectado(grupo_con_relleno):
                                combi_final = combi_candidato
                                union_geom = (
                                    grupo_con_relleno.geometry.iloc[0]
                                    if len(combi_final) == 1
                                    else grupo_con_relleno.geometry.union_all()
                                )

                    centroide = union_geom.centroid
                    distancia_al_punto = centroide.distance(punto_missing)

                    if distancia_al_punto < tolerancia and distancia_al_punto < mejor_distancia:
                        mejor_resultado = {
                            "zonas_ids": [zona_id],
                            "zona_principal": zona_id,
                            "segmentos_ids": combi_final,
                            "tamaño_grupo": len(combi_final),
                            "distancia": distancia_al_punto,
                            "geometry": union_geom,
                            "centroide": centroide,
                            "zonas_encontradas": 1,
                        }
                        mejor_distancia = distancia_al_punto

                except Exception:
                    continue

        if mejor_resultado is not None:
            resultados.append(mejor_resultado)
            zonas_restantes = zonas_restantes[~zonas_restantes["ID"].isin(mejor_resultado["zonas_ids"])]
            segmentos_restantes = segmentos_restantes.drop(index=mejor_resultado["segmentos_ids"])

            if verbose:
                print(f"    -> {mejor_resultado['tamaño_grupo']} segmentos, "
                      f"dist={mejor_resultado['distancia']:.1f}m")

    if verbose:
        print(f"\nPolígonos creados: {len(resultados)}")
        print(f"Segmentos usados: {len(segmentos_disponibles) - len(segmentos_restantes)}")
        print(f"Zonas sin resolver: {len(zonas_restantes)}")

    return resultados, segmentos_restantes, zonas_restantes

In [8]:
TOLERANCIA_1 = 30  # metros

resultados_iter_1, segmentos_restantes_1, zonas_restantes_1 = ejecutar_algoritmo_iterativo(segmentos_disponibles, zonas_missing, tolerancia=TOLERANCIA_1)

  procesando zona 1/229 - segmentos restantes: 989


    -> 1 segmentos, dist=0.6m
    -> 3 segmentos, dist=17.1m
    -> 2 segmentos, dist=1.0m
    -> 3 segmentos, dist=1.0m
    -> 5 segmentos, dist=0.6m
    -> 7 segmentos, dist=3.3m
    -> 3 segmentos, dist=1.7m
    -> 1 segmentos, dist=3.5m
    -> 3 segmentos, dist=1.9m
    -> 3 segmentos, dist=1.1m
    -> 3 segmentos, dist=1.8m
    -> 1 segmentos, dist=3.2m
    -> 1 segmentos, dist=1.3m
    -> 2 segmentos, dist=20.1m
    -> 1 segmentos, dist=10.7m
    -> 1 segmentos, dist=0.8m
    -> 3 segmentos, dist=3.4m
    -> 1 segmentos, dist=2.6m
    -> 2 segmentos, dist=2.4m
    -> 6 segmentos, dist=15.2m
    -> 1 segmentos, dist=1.7m
    -> 2 segmentos, dist=1.8m
    -> 3 segmentos, dist=0.7m
    -> 3 segmentos, dist=9.0m
    -> 2 segmentos, dist=2.8m
    -> 10 segmentos, dist=1.4m
    -> 2 segmentos, dist=1.1m
    -> 7 segmentos, dist=2.0m
    -> 4 segmentos, dist=3.2m
  procesando zona 51/229 - segmentos restantes: 903
    -> 1 segmentos, dist=1.0m
    -> 2 segmentos, dist=7.5m
    -> 6 segm

## 3. Segunda pasada, con tolerancia más alta

Se reutiliza la misma función sobre lo que quedó sin resolver en la primera pasada (`segmentos_restantes_1`, `zonas_restantes_1`), aflojando la tolerancia a 50 metros.

In [9]:
TOLERANCIA_2 = 50  # metros

resultados_iter_2, segmentos_restantes_final, zonas_restantes_final = ejecutar_algoritmo_iterativo(segmentos_restantes_1, zonas_restantes_1, tolerancia=TOLERANCIA_2)

  procesando zona 1/89 - segmentos restantes: 512
    -> 2 segmentos, dist=46.6m
    -> 2 segmentos, dist=47.6m
    -> 3 segmentos, dist=39.1m
    -> 2 segmentos, dist=32.2m
    -> 2 segmentos, dist=40.2m
    -> 3 segmentos, dist=41.8m
    -> 5 segmentos, dist=31.0m
    -> 1 segmentos, dist=41.6m
    -> 1 segmentos, dist=46.4m
    -> 3 segmentos, dist=41.5m
    -> 3 segmentos, dist=40.1m
  procesando zona 51/89 - segmentos restantes: 485
    -> 5 segmentos, dist=23.8m
    -> 6 segmentos, dist=42.9m
    -> 3 segmentos, dist=41.6m
    -> 2 segmentos, dist=37.4m
    -> 6 segmentos, dist=36.9m
    -> 15 segmentos, dist=24.5m
    -> 1 segmentos, dist=39.6m

Polígonos creados: 18
Segmentos usados: 65
Zonas sin resolver: 71


### Unir resultados y guardar

Se combinan las zonas resueltas por Voronoi y por las dos pasadas del algoritmo iterativo en un único GeoDataFrame, con una columna `metodo` para poder distinguir el origen de cada polígono.

In [10]:
import os
os.makedirs("output", exist_ok=True)

# --- Zonas resueltas por Voronoi ---
zonas_voronoi_final = zonas_voronoi.loc[zonas_encontradas_voronoi["index_right"].unique()].copy()
zonas_voronoi_final = zonas_voronoi_final.drop(columns=["centroide_zona"], errors="ignore")
zonas_voronoi_final["metodo"] = "voronoi"

# --- Zonas resueltas por el algoritmo iterativo (ambas pasadas) ---
def resultados_a_gdf(resultados, metodo, crs):
    filas = [{"geometry": r["geometry"], "metodo": metodo,
              "tamaño_grupo": r["tamaño_grupo"], "distancia": r["distancia"],
              "zona_principal": r["zona_principal"]} for r in resultados]
    return gpd.GeoDataFrame(filas, crs=crs)

gdf_iter_1 = resultados_a_gdf(resultados_iter_1, f"iterativo_{TOLERANCIA_1}m", CRS_METROS)
gdf_iter_2 = resultados_a_gdf(resultados_iter_2, f"iterativo_{TOLERANCIA_2}m", CRS_METROS)

# --- Combinar todo ---
columnas_comunes = ["geometry", "metodo"]
todas_las_zonas = pd.concat([
    zonas_voronoi_final[columnas_comunes],
    gdf_iter_1[columnas_comunes] if len(gdf_iter_1) else gpd.GeoDataFrame(columns=columnas_comunes),
    gdf_iter_2[columnas_comunes] if len(gdf_iter_2) else gpd.GeoDataFrame(columns=columnas_comunes)], ignore_index=True)
todas_las_zonas = gpd.GeoDataFrame(todas_las_zonas, crs=CRS_METROS)

output_path = "../output/zonas_finales_rerun.gpkg"
todas_las_zonas.to_file(output_path, driver="GPKG")

total_encontradas = len(zonas_voronoi_final) + len(gdf_iter_1) + len(gdf_iter_2)
print(f"Total de zonas resueltas: {total_encontradas} de {len(zonas)} "
      f"({total_encontradas/len(zonas)*100:.1f}%)")
print(f"  - Voronoi: {len(zonas_voronoi_final)}")
print(f"  - Iterativo {TOLERANCIA_1}m: {len(gdf_iter_1)}")
print(f"  - Iterativo {TOLERANCIA_2}m: {len(gdf_iter_2)}")
print(f"  - Zonas aún sin resolver: {len(zonas_restantes_final)}")
print(f"\nGuardado en: {output_path}")

Total de zonas resueltas: 527 de 598 (88.1%)
  - Voronoi: 369
  - Iterativo 30m: 140
  - Iterativo 50m: 18
  - Zonas aún sin resolver: 71

Guardado en: ../output/zonas_finales_rerun.gpkg


## 4. Mapa final

Una sola visualización con todas las capas: zonas Voronoi, zonas del algoritmo iterativo (por tolerancia), segmentos sin asignar y zonas que quedaron sin resolver.

In [ ]:
todas_las_zonas_wgs = todas_las_zonas.to_crs(epsg=4326)
segmentos_restantes_wgs = (segmentos_restantes_final.drop(columns=["centroide"], errors="ignore").to_crs(epsg=4326) if len(segmentos_restantes_final) else None)
zonas_restantes_wgs = zonas_restantes_final.to_crs(epsg=4326) if len(zonas_restantes_final) else None

centro = todas_las_zonas_wgs.geometry.union_all().centroid
mapa = folium.Map(location=[centro.y, centro.x], zoom_start=12, tiles="OpenStreetMap")

colores_por_metodo = {
    "voronoi": {"fillColor": "purple", "color": "purple"},
    f"iterativo_{TOLERANCIA_1}m": {"fillColor": "green", "color": "darkgreen"},
    f"iterativo_{TOLERANCIA_2}m": {"fillColor": "orange", "color": "darkorange"},
}

for metodo, colores in colores_por_metodo.items():
    capa = todas_las_zonas_wgs[todas_las_zonas_wgs["metodo"] == metodo]
    if len(capa) == 0:
        continue
    folium.GeoJson(
        capa,
        name=f"{metodo} ({len(capa)})",
        style_function=lambda x, c=colores: {
            "fillColor": c["fillColor"], "color": c["color"],
            "weight": 1, "fillOpacity": 0.4,
        },
    ).add_to(mapa)

if segmentos_restantes_wgs is not None and len(segmentos_restantes_wgs):
    folium.GeoJson(
        segmentos_restantes_wgs,
        name=f"Segmentos sin asignar ({len(segmentos_restantes_wgs)})",
        style_function=lambda x: {
            "fillColor": "red", "color": "darkred", "weight": 1, "fillOpacity": 0.6,
        },
    ).add_to(mapa)

if zonas_restantes_wgs is not None and len(zonas_restantes_wgs):
    for _, row in zonas_restantes_wgs.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=5, color="gray", fill=True, fillColor="lightgray", fillOpacity=0.8,
            popup=f"Zona sin resolver: {row['ID']}", tooltip="Zona sin resolver",
        ).add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa.save("output/mapa_zonas_finales.html")
mapa

## 5. Armado de zonas a mano

Si bien el algoritmo iterativo permitió resolver la mayor parte de las zonas, persistieron casos puntuales que no podían corregirse de forma automática sin  incurrir en un costo de desarrollo desproporcionado respecto a la cantidad de casos afectados. Por este motivo, se optó por resolver estos casos de forma manual en QGIS, en lugar de seguir ajustando los parámetros o la lógica del algoritmo.

Los segmentos asignados erróneamente por el algoritmo, detectables a simple vista, y los segmentos que quedaron sin asignar a ninguna zona, se resolvieron manualmente en QGIS.

Adicionalmente, se identificaron las zonas que quedaron asociadas a más de una sección censal a la vez. Por definición, cada polígono de la zonificación debe pertenecer a una única sección censal, por lo que estos casos también se corrigieron manualmente en QGIS.

Por último, tras petición del equipo, se amplificó la zona de interés sumando los departamentos de: Soriano, Flores, Florida, Maldonado, Lavalleja, Rocha y San José (dividido en algunas zonas). 

In [9]:
zonas_ampliado = gpd.read_file("../data/zonas-estimadas-ampliado.gpkg")
secciones_censales = gpd.read_file("../data/sc_23_pg.gpkg")
segmentos_censales_completo = gpd.read_file("../data/sg_23_pg.gpkg")

In [ ]:
crs_metros = 32721
zonas_ampliado = zonas_ampliado.to_crs(epsg=crs_metros)
secciones_censales = secciones_censales.to_crs(epsg=crs_metros)

zonas_ampliado = zonas_ampliado.drop(columns=["CODSECC"], errors="ignore")
# Achico 2 metros cada zona para ignorar el ruido de los bordes
zonas_achicadas = zonas_ampliado.copy()
zonas_achicadas["geometry"] = zonas_achicadas.geometry.buffer(-2)

asignacion = gpd.sjoin(zonas_achicadas, secciones_censales[["CODSECC", "geometry"]], how="left", predicate="within")
zonas_resueltas_idx = asignacion.dropna(subset=["CODSECC"]).index
zonas_en_multiples_secciones = zonas_ampliado[~zonas_ampliado.index.isin(zonas_resueltas_idx)]

print(f"Zonas que caen en más de una sección censal: {len(zonas_en_multiples_secciones)} de {len(zonas_ampliado)}")
zonas_en_multiples_secciones

In [ ]:
zonas_conflictivas_wgs = zonas_en_multiples_secciones.to_crs(epsg=4326)
secciones_censales_wgs = secciones_censales.to_crs(epsg=4326)

# Identificar las secciones censales que intersectan las zonas conflictivas
intersecciones = gpd.sjoin(zonas_en_multiples_secciones[["geometry"]],secciones_censales[["CODSECC", "geometry"]], how="inner", predicate="intersects")

# Secciones involucradas
secciones_involucradas_codigos = intersecciones["CODSECC"].unique()
secciones_involucradas_wgs = secciones_censales_wgs[secciones_censales_wgs["CODSECC"].isin(secciones_involucradas_codigos)]

centro = zonas_conflictivas_wgs.geometry.union_all().centroid
mapa_conflictos = folium.Map(location=[centro.y, centro.x], zoom_start=9,tiles="OpenStreetMap")

# Secciones censales involucradas (solo bordes)
folium.GeoJson(
    secciones_involucradas_wgs,
    name=f"Secciones censales involucradas ({len(secciones_involucradas_wgs)})",
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "blue",
        "weight": 2,
        "dashArray": "5, 5"}, tooltip=folium.GeoJsonTooltip(fields=["CODSECC"])).add_to(mapa_conflictos)

# Zonas conflictivas (relleno rojo)
folium.GeoJson(
    zonas_conflictivas_wgs,
    name=f"Zonas conflictivas ({len(zonas_conflictivas_wgs)})",
    style_function=lambda x: {
        "fillColor": "red",
        "color": "darkred",
        "weight": 1.5,
        "fillOpacity": 0.5}).add_to(mapa_conflictos)

folium.LayerControl().add_to(mapa_conflictos)
mapa_conflictos

Decidimos sólo modificar las zonas "Core", ya que las zonas "Periferia" corresponden a departamentos. Esta modificación fue realizada en QGIS, resultando en el archivo 'zonas-montevideo-actualizadas'.

In [3]:
CRS_METROS = 32721
zonificacion = gpd.read_file("../data/zonificacion-montevideo (2).gpkg").to_crs(epsg=CRS_METROS) # archivo con columna tipo_zona
file = gpd.read_file("../data/zonas-montevideo-actualizadas.gpkg").to_crs(epsg=CRS_METROS) # nueva zonificación
secciones_censales = gpd.read_file("../data/sc_23_pg.gpkg")

In [4]:
file_exploded = file.explode(index_parts=True).reset_index() # separar los multipolígonos en polígonos individuales
file_exploded["area_m2"] = file_exploded.geometry.area # calculo el área de cada polígono

file_exploded = file_exploded.sort_values(["level_0", "area_m2"], ascending=[True, False])
file_limpio = (file_exploded.groupby("level_0").first().reset_index(drop=True))
file_limpio = file_limpio[file_limpio["area_m2"] >= 1].reset_index(drop=True)

In [5]:
file_limpio["poligono_id"] = range(1, len(file_limpio) + 1)

core_boundary = zonificacion[zonificacion["tipo_zona"] == "Core"].geometry.union_all()

file_limpio["centroide"] = file_limpio.geometry.centroid
file_limpio["tipo_zona"] = file_limpio["centroide"].apply(
    lambda c: "Core" if core_boundary.contains(c) else "Periferia")
file_limpio = file_limpio.drop(columns=["centroide"])
file_limpio.loc[file_limpio["poligono_id"] == 468, "tipo_zona"] = "Core" # este poligono era la excepcion, porq su centroide está por fuera
print(file_limpio["tipo_zona"].value_counts())

tipo_zona
Core         618
Periferia     12
Name: count, dtype: int64


In [6]:
file_limpio = file_limpio[['poligono_id', 'tipo_zona', 'geometry']]

## 6. Asignación de departamento, sección censal y segmentos censales

Una vez finalizada la recreación de las zonas de Steer, se le asigna a cada polígono de la zonificación su departamento y sección censal.

El departamento se determina a partir del centroide de cada polígono. Para la sección censal, se consideran las secciones que intersectan al menos el 1% de la superficie del polígono. Para los segmentos censales, se considera un punto representativo.

Finalmente, se genera un mapa interactivo con la zonificación ajustada, las secciones censales y los segmentos censales.

In [7]:
file_limpio = file_limpio.set_crs(epsg=32721)
secciones_censales = secciones_censales.to_crs(epsg=32721)
# DEPARTAMENTO: uso el centroide
deptos = secciones_censales.dissolve(by="DEPTO",aggfunc="first")[["NOMDEPTO","geometry"]].reset_index()
deptos = deptos.rename(columns={"DEPTO":"depto_codigo", "NOMDEPTO":"depto_nombre"})
centroides = file_limpio[["poligono_id","geometry"]].copy()
centroides["geometry"]= centroides.geometry.centroid
asignacion_depto = gpd.sjoin(centroides, deptos, how="left", predicate="within").drop_duplicates("poligono_id")
resultado=file_limpio.merge(asignacion_depto[["poligono_id", "depto_codigo", "depto_nombre"]],on="poligono_id",how="left")
resultado.loc[resultado["poligono_id"]==468,"depto_codigo"]="01"
resultado.loc[resultado["poligono_id"]==468,"depto_nombre"]="MONTEVIDEO"

# SECCION CENSAL: usamos el porcentaje de area compartida
inter=gpd.overlay(resultado[["poligono_id","geometry"]],secciones_censales[["CODSECC","geometry"]],how="intersection",keep_geom_type=False)

def extraer_poligonos(geom):
    if geom.geom_type in ["Polygon","MultiPolygon"]:
        return geom
    if geom.geom_type=="GeometryCollection":
        polys=[g for g in geom.geoms if g.geom_type in ["Polygon","MultiPolygon"]]
        if polys:
            return gpd.GeoSeries(polys).union_all()
    return None

inter["geometry"]=inter.geometry.apply(extraer_poligonos)
inter=inter[inter.geometry.notna()]
inter["area_interseccion"]=inter.geometry.area
areas_zona=resultado[["poligono_id","geometry"]].copy()
areas_zona["area_zona"]= areas_zona.geometry.area
inter=inter.merge(areas_zona[["poligono_id","area_zona"]],on="poligono_id",how="left")
inter["porcentaje"]= inter["area_interseccion"]/inter["area_zona"]
inter = inter[inter["porcentaje"] >= 0.01] # porcentaje 1%
codsecc_por_zona = inter.groupby("poligono_id")["CODSECC"].apply(lambda x:", ".join(sorted(set(x.astype(str))))).reset_index()
resultado = resultado.merge(codsecc_por_zona,on="poligono_id",how="left")
resultado = resultado.rename(columns={"CODSECC":"seccion_codigo"})
resultado=resultado.loc[:,~resultado.columns.duplicated()]

In [10]:
# SEGMENTOS CENSALES: uso un punto representativo

segmentos_codseg = segmentos_censales_completo[["CODSEG", "geometry"]].copy()
segmentos_codseg["punto_rep"] = segmentos_codseg.geometry.representative_point()
segmentos_codseg = segmentos_codseg.set_geometry("punto_rep", crs=segmentos_censales_completo.crs).to_crs(resultado.crs)

segmentos_en_zona = gpd.sjoin(segmentos_codseg[["CODSEG", "punto_rep"]].set_geometry("punto_rep"), resultado[["poligono_id", "geometry"]], how="left", predicate="within")

# Chequeo de duplicados, un mismo CODSEG no se debería repetir en más de una zona
duplicados = segmentos_en_zona.dropna(subset=["poligono_id"]).groupby("CODSEG")["poligono_id"].nunique()
duplicados = duplicados[duplicados > 1]
if len(duplicados):
    print(f"Aviso: {len(duplicados)} CODSEG aparecen asignados a más de una zona final")

# Agrupamos los CODSEG por zona
codseg_por_zona = segmentos_en_zona.dropna(subset=["poligono_id"]).groupby("poligono_id")["CODSEG"].apply(lambda x: ", ".join(sorted(set(x.astype(str))))).reset_index()

resultado = resultado.drop(columns="CODSEG", errors="ignore")
resultado = resultado.merge(codseg_por_zona, on="poligono_id", how="left")
resultado["CODSEG"] = resultado["CODSEG"].fillna("")

zonas_sin_codseg = (resultado["CODSEG"] == "").sum()
print(f"Zonas finales sin ningún CODSEG asignado: {zonas_sin_codseg} de {len(resultado)}")
print(f"Total de CODSEG distintos asignados a alguna zona: {segmentos_en_zona.dropna(subset=['poligono_id'])['CODSEG'].nunique()} de {segmentos_censales_completo['CODSEG'].nunique()}")

Zonas finales sin ningún CODSEG asignado: 0 de 630
Total de CODSEG distintos asignados a alguna zona: 3156 de 4538


In [ ]:
# MAPA
resultado_wgs=resultado.to_crs(epsg=4326)
secciones_wgs=secciones_censales.to_crs(epsg=4326)
segmentos_wgs = segmentos_censales_completo.to_crs(epsg=4326)
centro=resultado_wgs.geometry.union_all().centroid
m2=folium.Map(location=[centro.y,centro.x],zoom_start=9,tiles="OpenStreetMap")

folium.GeoJson(segmentos_wgs,name = "Segmentos censales", style_function=lambda feature:{"fillColor":"#FFA865","color":"#FFA865","weight":1,"fillOpacity":0},tooltip=folium.GeoJsonTooltip(fields=["CODSEG"],aliases=["CODSEG:"],sticky=False)).add_to(m2)
folium.GeoJson(secciones_wgs,name = "Secciones censales", style_function=lambda feature:{"fillColor":"transparent","color":"#002244","weight":1,"fillOpacity":0},tooltip=folium.GeoJsonTooltip(fields=["DEPTO","CODSECC"],aliases=["Departamento:","CODSECC:"],sticky=False)).add_to(m2)
folium.GeoJson(resultado_wgs,name = "Zonificación", style_function=lambda feature:{"fillColor":"steelblue","color":"#009FDA","weight":1,"fillOpacity":0.6},highlight_function=lambda x:{"weight":3,"color":"yellow","fillOpacity":0.8},tooltip=folium.GeoJsonTooltip(fields=["poligono_id","tipo_zona","depto_codigo","depto_nombre","seccion_codigo", "CODSEG"],aliases=["ID:","Tipo zona:","Código depto:","Departamento:","Sección censal:","Segmentos Censales:"],sticky=False)).add_to(m2)

folium.LayerControl().add_to(m2)
m2

In [ ]:
# MAPA PARA VER ZONAS CORE Y PERIFERIA

zonificacion_wgs = resultado.to_crs(epsg=4326)
centro = zonificacion_wgs.geometry.union_all().centroid
mapa = folium.Map(location=[centro.y, centro.x], zoom_start=10, tiles="OpenStreetMap")

tipo_zona = folium.FeatureGroup(name="Tipo de zona (Core/Periferia)", show=True)
colores_tipo = {"Core": "purple", "Periferia": "teal"}

for tipo, color in colores_tipo.items():
    capa = zonificacion_wgs[zonificacion_wgs["tipo_zona"] == tipo]

    folium.GeoJson(capa,
        style_function=lambda x, c=color: {
            "fillColor": c,
            "color": c,
            "weight": 1,
            "fillOpacity": 0.35}).add_to(tipo_zona)
tipo_zona.add_to(mapa)

grupo_zonificacion = folium.FeatureGroup(name="Zonificación", show=True)

folium.GeoJson(
    zonificacion_wgs,
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "black",
        "weight": 1,
        "fillOpacity": 0},
    highlight_function=lambda x: {
        "weight": 3,
        "color": "yellow",
        "fillOpacity": 0.3},
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "poligono_id",
            "tipo_zona"],
        aliases=[
            "ID polígono:",
            "Tipo de zona:"],
        sticky=False
    )).add_to(grupo_zonificacion)

grupo_zonificacion.add_to(mapa)

folium.LayerControl(collapsed=False).add_to(mapa)
mapa

In [12]:
resultado = resultado[["poligono_id","tipo_zona","geometry","depto_codigo","depto_nombre","seccion_codigo","CODSEG"]]
resultado.to_file("../output/zonas-montevideo-actualizadas-steer.gpkg", driver="GPKG")